<a href="https://colab.research.google.com/github/ManeSayali/Fraud-bank-account-Detection/blob/main/fraud_detection_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Importing required libraries**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.metrics import roc_auc_score,matthews_corrcoef
import numpy as np
from imblearn.under_sampling import NearMiss
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import GaussianNB
#import joblib
from sklearn.metrics import (f1_score, roc_auc_score,confusion_matrix, accuracy_score,
                             precision_score, recall_score)
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dataset/dataset_final.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
df.shape

In [ ]:
df["fraud_bool"].value_counts()

Here we can see that the target colum ie "fraud_bool" having non_fraud(0.0) is majority instance and fraud(1.0) is minority instance in resultant colum.
that means the data set is highly **Imbalance**

In [ ]:
X=df.drop(["fraud_bool"],axis=1)
Y=df["fraud_bool"]


our data is highly imbalance so we used Nearmiss techinique to balance the data.

---



---




**NearMiss** is a family of undersampling methods that select examples from the majority class based on their (Euclidean) distance from examples in the minority class. The basic idea is to keep examples that are close to the minority class and discard examples that are far away from them.

In [ ]:
nm = NearMiss()
x_sample, y_sample= nm.fit_resample(X,Y)
print(y_sample.value_counts())

**Spliting dataset**

Here we spliting dataset for training and testing purpose
training dataset is 80% of full dataset and Testing dataset is 20% of full dataset

In [ ]:
x_train_sample, x_test_sample, y_train_sample, y_test_sample = train_test_split(x_sample, y_sample, test_size=0.2,random_state=42,stratify=y_sample)

In [ ]:
print(x_train_sample.shape)
print(x_test_sample.shape)

In [ ]:
weights_assigned={0:1,1:100}

In [ ]:
n_inputs=(len(x_train_sample.columns))

## **ANN**

In [ ]:
model = keras.Sequential()
model.add(keras.layers.BatchNormalization(input_shape=[x_train_sample.shape[1]]))
model.add(keras.layers.Dense(512,input_dim=n_inputs,activation='relu',kernel_initializer='he_uniform'))#128
#kernal_initializer='he_uniform' ---> weight initializer for relu
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(128,activation='relu'))#64
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(64,activation='relu'))#64
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(64,activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(32,activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(1,activation='sigmoid'))


model.compile(loss='binary_crossentropy',optimizer="Adam")#tf.keras.optimizers.Adam(learning_rate=0.007)
model.fit(x_train_sample,y_train_sample,epochs=100,batch_size=1032)#,class_weight=weights_assigned

In [ ]:
y_pred=model.predict(x_test_sample)

In [ ]:
 print("roc_auc_score :",roc_auc_score(y_test_sample,y_pred) *100 ,"%")

In [ ]:
y_pred01=np.round(y_pred)
y_pred01=y_pred01.astype(int)
print(classification_report(y_test_sample, y_pred01))
print(confusion_matrix(y_test_sample,y_pred01))

In [ ]:
mcc=matthews_corrcoef(y_test_sample,y_pred01)
print(mcc)

## **SVM**

In [ ]:
from sklearn.svm import SVC

In [ ]:
svm= SVC(kernel='linear', random_state=42)
svm.fit(x_train_sample, y_train_sample)


pred_02= svm.predict(x_test_sample)
print(classification_report(y_test_sample,pred_02))
print("roc auc:",roc_auc_score(y_test_sample,pred_02))


In [ ]:
print(confusion_matrix(y_test_sample,pred_02))

In [ ]:
mcc=matthews_corrcoef(y_test_sample,pred_02)
print(mcc)

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(svm,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

## **RANDOM FOREST**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=200, criterion="entropy")
rf_clf.fit(x_train_sample, y_train_sample)
rf_clf.estimators_[0]

In [ ]:
y_pred1=rf_clf.predict(x_test_sample)

In [ ]:
cr1=classification_report(y_test_sample,y_pred1)
print(cr1)
print(roc_auc_score(y_test_sample,y_pred1))

In [ ]:
mcc=matthews_corrcoef(y_test_sample,y_pred1)
print(mcc)
print(confusion_matrix(y_test_sample,y_pred1))

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(rf_clf,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

In [ ]:
rf_clf.feature_importances_
x_train_sample.columns

feature_importances = pd.Series(rf_clf.feature_importances_, index=x_train_sample.columns)
feature_importances.sort_values(ascending=False, inplace=True)
feature_importances

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(8,4), dpi=100)
feature_importances.plot.bar()
plt.title("Feature importances")
plt.xlabel("Features")
plt.ylabel("Mean Decrease in Impurity")
plt.show()

## **XGboost**

In [ ]:
from xgboost import XGBClassifier
xgb1 = XGBClassifier(random_state=42)
xgb1.fit(x_train_sample, y_train_sample)
y_pred03=xgb1.predict(x_test_sample)

In [ ]:
print(classification_report(y_test_sample,y_pred03))
print("roc auc:",roc_auc_score(y_test_sample,y_pred03))

In [ ]:
mcc=matthews_corrcoef(y_test_sample,y_pred03)
print(mcc)
print(confusion_matrix(y_test_sample,y_pred03))

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(xgb1,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

## **GB**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
Gb1 = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=42)
Gb1.fit(x_train_sample, y_train_sample)
y_pred04=Gb1.predict(x_test_sample)


In [ ]:
print(classification_report(y_test_sample,y_pred04))
print("roc auc:",roc_auc_score(y_test_sample,y_pred04))

In [ ]:
mcc=matthews_corrcoef(y_test_sample,y_pred04)
print("mcc:",mcc)
print(confusion_matrix(y_test_sample,y_pred04))

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(Gb1,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

# **logistic regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
LR1 = LogisticRegression(random_state=42,max_iter=1000)
LR1.fit(x_train_sample, y_train_sample)
y_pred05=LR1.predict(x_test_sample)

In [ ]:
print(classification_report(y_test_sample,y_pred05))
print("roc auc:",roc_auc_score(y_test_sample,y_pred05))
mcc=matthews_corrcoef(y_test_sample,y_pred05)
print("mcc:",mcc)
print(confusion_matrix(y_test_sample,y_pred05))

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(LR1,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

# **naive bayes**

In [ ]:
clf_GB = GaussianNB()

# train the classifier on the training set
clf_GB.fit(x_train_sample, y_train_sample)
y_pred06 = clf_GB.predict(x_test_sample)

In [ ]:
print(classification_report(y_test_sample,y_pred06))
print("roc auc:",roc_auc_score(y_test_sample,y_pred06))
mcc=matthews_corrcoef(y_test_sample,y_pred06)
print("mcc:",mcc)
print(confusion_matrix(y_test_sample,y_pred06))

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(clf_GB,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

# **decision tree**

In [ ]:
DT = DecisionTreeClassifier( max_depth=1, random_state=42)

# train the classifier on the training set
DT.fit(x_train_sample, y_train_sample)

# make predictions on the testing set for Confusion Matrix
y_pred07 = DT.predict(x_test_sample)
confusion_mat = confusion_matrix(y_test_sample, y_pred07)

In [ ]:
print(classification_report(y_test_sample,y_pred07))
print("roc auc:",roc_auc_score(y_test_sample,y_pred07))
mcc=matthews_corrcoef(y_test_sample,y_pred07)
print("mcc:",mcc)
print(confusion_matrix(y_test_sample,y_pred07))

In [ ]:
y_tr = y_train_sample

y_pr = cross_val_predict(DT,x_train_sample,y_train_sample, cv=5)
print(confusion_matrix(y_tr, y_pr))
print('Accuracy: ', accuracy_score(y_tr,y_pr))

After sampling the the data we try 8 different models to predict the fraud account.  ANN is one of the model to work very good on given dataset which having high F1-score

In [ ]:
#joblib.dump(model,'/content/drive/MyDrive/ANN_02.joblib')
#pickle.dump(model, open('model_ANN.pkl', 'wb'))
#import pickle